# 02 — Union-based SQL Injection

> **`vuln_class`:** `SQL_INJ_UNION` · **Риск:** 9/10 · **CWE-89** · **CAPEC-66 (Union variant)**

Это **вариант** SQL Injection, где атакующий через `UNION SELECT` к легитимному запросу присоединяет свой — и читает данные из таблиц, которые не были предусмотрены.


## 🧒 Аналогия для ребёнка

Представь, ты учительница и проверяешь домашки. Тебе сдают
одну тетрадь — ты её читаешь и ставишь оценку.

А теперь представь, что хитрый ученик **склеил скотчем** свою
тетрадь и тетрадь соседа — и сдал тебе **двойную тетрадь**.
Ты читаешь обе подряд, ставишь оценку — но видишь и чужие
ответы тоже.

UNION SELECT — это «скотч» в SQL. Атакующий **подклеивает второй
запрос** к твоему и получает данные «соседней тетради» —
например, таблицы с паролями.


## 1. Setup — поиск по каталогу товаров + соседняя таблица auth.users

У нас есть публичная таблица товаров и приватная таблица учёток.


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Создаёт две таблицы: products (публичная) и users (приватная).
def setup_shop_db():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE products (
            id    INTEGER PRIMARY KEY,
            title TEXT,
            price REAL
        )""")
    cur.execute("""
        CREATE TABLE users (
            id            INTEGER PRIMARY KEY,
            login         TEXT,
            password_hash TEXT
        )""")
    cur.executemany(
        "INSERT INTO products (title, price) VALUES (?, ?)",
        [("Молоток", 500), ("Гвозди", 50), ("Дрель", 5000), ("Шуруповёрт", 7000)],
    )
    cur.executemany(
        "INSERT INTO users (login, password_hash) VALUES (?, ?)",
        [
            ("admin",  "hash_super_secret_admin"),
            ("ceo",    "hash_super_secret_ceo"),
            ("intern", "hash_intern_qwerty"),
        ],
    )
    conn.commit()
    return conn


conn = setup_shop_db()
section("Публичный каталог товаров (видим в UI)")
show_result(conn.execute("SELECT id, title, price FROM products").fetchall())

section("Приватная таблица учёток (НИКОГДА не должна попасть в UI)")
show_result(conn.execute("SELECT id, login, password_hash FROM users").fetchall())


## 2. Уязвимая функция поиска товара

Типичный поиск по подстроке — без параметризации.


In [ ]:
##
# @brief УЯЗВИМАЯ функция поиска товара по части названия.
# @param q  Подстрока запроса от пользователя.
# @return   Список товаров (id, title).
# @warning  Конкатенация → возможен UNION-based injection.
def search_products_BAD(conn, q: str):
    sql = f"SELECT id, title FROM products WHERE title LIKE '%{q}%'"
    print(f"  SQL: {sql}")
    return conn.execute(sql).fetchall()


section("Нормальный поиск")
show_result(search_products_BAD(conn, "Дре"))


## 3. Атака — поэтапная разведка через UNION

Атакующий не знает заранее, сколько колонок возвращает запрос
и какие у них типы. Подбирает шаг за шагом.


In [ ]:
section("ШАГ 1 атаки: подбираем число колонок")
# Если число колонок не сошлось — SQLite вернёт ошибку или 0 строк
for n in range(1, 5):
    payload = "x' UNION SELECT " + ", ".join(["NULL"] * n) + " --"
    try:
        rows = search_products_BAD(conn, payload)
        print(f"  Попытка {n} колонок: {len(rows)} строк ✓")
    except Exception as e:
        print(f"  Попытка {n} колонок: ОШИБКА — {e}")


section("ШАГ 2 атаки: подобрали число (2) — теперь крадём таблицу users")
payload = "x' UNION SELECT login, password_hash FROM users --"
print(f"  payload = {payload!r}")
rows = search_products_BAD(conn, payload)
print(f"\n  💀 Получено {len(rows)} строк, и среди них — все пароли:")
show_result(rows)


section("ШАГ 3 атаки: ещё страшнее — узнаём, какие вообще есть таблицы")
payload = "x' UNION SELECT name, sql FROM sqlite_master --"
print(f"  payload = {payload!r}")
rows = search_products_BAD(conn, payload)
print(f"\n  💀 Атакующий теперь знает всю схему БД:")
show_result(rows)


## 4. Аудитор Phase 1 — правило R005-union-suspicious

Используем `pglast` (в проде) или регулярки + AST (здесь — упрощённо).
Сигналы для подозрения:

1. `UNION` с **`NULL, NULL, NULL`** — probe-паттерн.
2. `UNION` с обращением к системной таблице (`sqlite_master`, `information_schema`, `pg_catalog`).
3. Рассинхрон числа колонок верх/низ (часто отлажен этап probe).
4. UNION-подзапрос, который повторяет литерал из WHERE верхнего (mirror).


In [ ]:
import sqlite3 as _sql


SUSPICIOUS_TABLES = {"sqlite_master", "information_schema", "pg_catalog",
                     "pg_user", "pg_authid", "users", "credentials"}


##
# @brief Phase 1 правило R005 — детект подозрительного UNION.
# @details
#   Здесь, для наглядности, мы запускаем регулярки на тексте SQL,
#   который функция СФОРМИРОВАЛА. В проде — обходим AST после pglast.parse_sql,
#   проверяем SelectStmt.op == SETOP_UNION и его правую часть.
def audit_R005_union(sql_text: str):
    findings = []
    # 1. NULL-only probe
    if re.search(r"UNION\s+(ALL\s+)?SELECT\s+(NULL\s*,?\s*)+(\-\-|$)",
                 sql_text, re.IGNORECASE):
        findings.append({
            "rule_id":       "R005-union-suspicious",
            "vuln_class":    "SQL_INJ_UNION",
            "severity":      "high", "risk_score": 8,
            "message":       "UNION SELECT NULL,NULL,... — probe-паттерн",
            "evidence_refs": ["CWE-89", "CAPEC-66"],
        })
    # 2. UNION с обращением к системной таблице
    m = re.search(r"UNION\s+(?:ALL\s+)?SELECT\s+.*?FROM\s+(\w+)",
                  sql_text, re.IGNORECASE | re.DOTALL)
    if m and m.group(1).lower() in SUSPICIOUS_TABLES:
        findings.append({
            "rule_id":       "R005-union-suspicious",
            "vuln_class":    "SQL_INJ_UNION",
            "severity":      "high", "risk_score": 9,
            "message":       f"UNION SELECT ... FROM {m.group(1)} — доступ к чувствительной таблице",
            "evidence_refs": ["CWE-89", "CAPEC-66"],
        })
    return findings


section("Аудитор анализирует SQL, который слепили из payload-а")
malicious_sql = "SELECT id, title FROM products WHERE title LIKE '%x' UNION SELECT login, password_hash FROM users --%'"
for f in audit_R005_union(malicious_sql):
    print_finding(f)


## 5. Безопасная функция — параметризация

Тот же поиск, но через `?`. Что бы атакующий ни прислал — это
будет строка, в которую LIKE поищет литерально.


In [ ]:
##
# @brief Безопасная функция: параметризация.
def search_products_GOOD(conn, q: str):
    sql = "SELECT id, title FROM products WHERE title LIKE ?"
    return conn.execute(sql, (f"%{q}%",)).fetchall()


section("АТАКА UNION на безопасную версию")
payload = "x' UNION SELECT login, password_hash FROM users --"
rows = search_products_GOOD(conn, payload)
print(f"  ✅ Получено {len(rows)} строк. Атакующий ищет товар, в названии которого буквально \"{payload}\" — таких нет.")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/02-sql-injection-union/README.md](../problems/vulnerabilities/02-sql-injection-union/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/02-sql-injection-union/solutions.md](../problems/vulnerabilities/02-sql-injection-union/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
